# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [11]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    file = open(filename, "r")
    file.write(datetime.strftime("%d/%m/%y, %H:%M:%S") + text)
    file.close()

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    try:
        file = open(filename, "r")
        for line in file:
            print(line.strip())
        file.close()
    except FileNotFoundError:
        print("File not found")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    file = open(filename, "r")
    finds=[]
    for line in file:
        if keyword.upper() in line.upper():
            finds.append(line)  
    file.close()
    return finds

def count_entries(filename):
    """Count and return the number of journal entries."""
    file = open(filename, "r")
    sum=0
    for line in file:
        if len(line.strip()) > 0:
            sum+=1
    return sum

def run_journal():
    """Menu-driven journal interface."""
    # TODO: while True loop with menu [A]dd [R]ead [S]earch [C]ount [Q]uit
    # TODO: Route to the correct function based on user input
    # option = input(print("Choose Menu Option: [A]dd [R]ead [S]earch [C]ount [Q]uit"))
    # if option in ['A','R','S','C','Q']:



# --- QUICK TEST (non-interactive) ---
# add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
# add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
# read_journal(JOURNAL_FILE)
# print("Search results:", search_journal(JOURNAL_FILE, "python"))
# print("Total entries:", count_entries(JOURNAL_FILE))

# run_journal()  # Uncomment to test the interactive menu

---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [10]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    try:
        with open(filename, 'r') as file:
            j = json.load(file)
        return j
    except FileNotFoundError:
        print("File not found")
    except json.JSONDecodeError:
        print("Unable to decode file")
    return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    with open(filename, 'w') as file:
        json.dump(tasks, file, indent=2)
    pass

def add_task(filename, tasks):
    """Prompt for title and priority, append new task."""
    title=input()
    priority=input()
    input_tasks={'title': title, 'priority':priority}
    with open(filename, "r") as file:
        json.dump(input_tasks, file, indent=2)

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    for task in tasks:
        print(f"title: {task.title} priority: {task.priority}")

def mark_done(tasks, idx):
    """Ask for a task index and mark it as done."""
    updated_tasks = []
    for iter, task in enumerate(tasks):
        if idx == iter:
            task["done"] = True
            updated_tasks.append(task)
    return updated_tasks
            

def delete_task(tasks, idx):
    """Ask for a task index and remove the task."""
    updated_tasks = []
    for iter, task in enumerate(tasks):
        if idx == iter:
            pass
        else:
            task["done"] = True
            updated_tasks.append(task)
    return updated_tasks

def run_todo():
    """Main menu loop with auto-save."""
    # TODO: Load tasks at start
    # TODO: while True menu: [A]dd [L]ist [D]one [R]emove [Q]uit
    # TODO: Save after every modification
    pass


# --- QUICK TEST (non-interactive) ---
# tasks = load_tasks(TODO_FILE)
# tasks.append({"title": "Test task", "priority": "high", "done": False})
# save_tasks(TODO_FILE, tasks)
# tasks = load_tasks(TODO_FILE)
# list_tasks(tasks)

# run_todo()  # Uncomment to test the interactive menu

---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [8]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [9]:
import csv

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    expenses = []
    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                category = (row.get("category") or "").strip()
                try:
                    amount = float(row.get("amount", ""))
                except (TypeError, ValueError):
                    continue
                if not category:
                    continue
                clean_row = {key: (value.strip() if isinstance(value, str) else value)
                             for key, value in row.items() if key is not None}
                clean_row["category"] = category
                clean_row["amount"] = amount
                expenses.append(clean_row)
    except FileNotFoundError:
        print(f"File not found: {filename}")
    return expenses

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    totals = {}
    for expense in expenses:
        category = expense["category"]
        totals[category] = totals.get(category, 0.0) + expense["amount"]
    return totals

def most_expensive(expenses):
    """Return the expense dict with the highest amount, or None if empty."""
    return max(expenses, key=lambda expense: expense["amount"], default=None)

def print_report(expenses):
    """Print a formatted expense summary report."""
    total = sum(expense["amount"] for expense in expenses)
    print(f"Total spent: ${total:.2f}")
    print("Spending by category:")
    for category, amount in sorted(total_by_category(expenses).items()):
        print(f"  {category}: ${amount:.2f}")
    highest = most_expensive(expenses)
    if highest is None:
        print("Most expensive item: none (no valid expenses)")
    else:
        print(f"Most expensive item: {highest['description']} ({highest['category']}) â€” ${highest['amount']:.2f}")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)


Total spent: $510.13
Spending by category:
  Education: $72.49
  Entertainment: $114.99
  Food: $162.75
  Health: $12.40
  Transport: $58.50
  Utilities: $89.00
Most expensive item: Electricity bill (Utilities) â€” $89.00


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [5]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [6]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:
        with open(filename, "r", encoding="utf-8") as file:
            data = json.load(file)
    except FileNotFoundError:
        report["errors"].append(f"File not found: {filename}")
        return report
    except json.JSONDecodeError as error:
        report["errors"].append(f"Invalid JSON: {error}")
        return report
    if not isinstance(data, list):
        report["errors"].append("JSON root must be a list")
        report["invalid"] = 1
        return report
    for index, item in enumerate(data):
        if not isinstance(item, dict):
            report["invalid"] += 1
            report["errors"].append(f"Record {index}: expected an object/dict")
            continue
        missing = [key for key in required_keys if key not in item]
        if missing:
            report["invalid"] += 1
            report["errors"].append(f"Record {index}: missing required keys {missing}")
        else:
            report["valid"] += 1
    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.reader(file)
            header = next(reader, None)
            if header is None:
                report["errors"].append("CSV file is empty; header is missing")
                report["invalid"] = 1
                return report
            expected_count = len(expected_columns) if not isinstance(expected_columns, int) else expected_columns
            if len(header) != expected_count:
                report["invalid"] += 1
                report["errors"].append(f"Header has {len(header)} columns; expected {expected_count}")
            elif not isinstance(expected_columns, int) and header != list(expected_columns):
                report["invalid"] += 1
                report["errors"].append(f"Header columns {header} do not match expected {list(expected_columns)}")
            for row_number, row in enumerate(reader, start=2):
                if len(row) != expected_count:
                    report["invalid"] += 1
                    report["errors"].append(f"Row {row_number} has {len(row)} fields; expected {expected_count}")
                else:
                    report["valid"] += 1
    except FileNotFoundError:
        report["errors"].append(f"File not found: {filename}")
    except csv.Error as error:
        report["errors"].append(f"CSV parsing error: {error}")
    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    print(f"Validation report for {filename}")
    print(f"  Valid: {report['valid']}  Invalid: {report['invalid']}")
    if report["errors"]:
        print("  Errors:")
        for error in report["errors"]:
            print(f"    - {error}")
    else:
        print("  No errors found.")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")


Validation report for good_data.json
  Valid: 3  Invalid: 0
  No errors found.
Validation report for bad_data.json
  Valid: 2  Invalid: 2
  Errors:
    - Record 1: missing required keys ['email']
    - Record 2: expected an object/dict
Validation report for corrupted.json
  Valid: 0  Invalid: 0
  Errors:
    - Invalid JSON: Expecting property name enclosed in double quotes: line 1 column 20 (char 19)
Validation report for good_data.csv
  Valid: 2  Invalid: 0
  No errors found.
Validation report for bad_data.csv
  Valid: 2  Invalid: 2
  Errors:
    - Row 3 has 2 fields; expected 3
    - Row 4 has 4 fields; expected 3


---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [3]:
import json

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")


✅ students.json created.


In [4]:
import json
import csv

def _load_record_list(json_filename):
    with open(json_filename, "r", encoding="utf-8") as file:
        data = json.load(file)
    if not isinstance(data, list):
        raise ValueError("JSON data must be a list of dictionaries")
    if any(not isinstance(record, dict) for record in data):
        raise ValueError("Every JSON record must be a dictionary")
    return data

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    try:
        records = _load_record_list(json_filename)
        if not records:
            print("No records to export; an empty CSV file was created.")
            open(csv_filename, "w", newline="", encoding="utf-8").close()
            return
        fieldnames = list(records[0].keys())
        if any(set(record.keys()) != set(fieldnames) for record in records):
            raise ValueError("JSON records must have matching fields")
        with open(csv_filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(records)
    except (FileNotFoundError, json.JSONDecodeError, ValueError) as error:
        print(f"Could not convert JSON to CSV: {error}")

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as file:
            records = list(csv.DictReader(file))
        with open(json_filename, "w", encoding="utf-8") as file:
            json.dump(records, file, indent=2)
    except (FileNotFoundError, csv.Error) as error:
        print(f"Could not convert CSV to JSON: {error}")

def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    try:
        records = _load_record_list(json_filename)
        filtered = [record for record in records if record.get(filter_key) == filter_value]
        fieldnames = list(records[0].keys()) if records else [filter_key]
        if any(set(record.keys()) != set(fieldnames) for record in filtered):
            raise ValueError("JSON records must have matching fields")
        with open(csv_filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(filtered)
    except (FileNotFoundError, json.JSONDecodeError, ValueError) as error:
        print(f"Could not export filtered CSV: {error}")


# --- TEST ---
# JSON â†’ CSV
json_to_csv("students.json", "students_export.csv")

# CSV â†’ JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
with open("students_roundtrip.json", "r") as f:
    roundtrip = json.load(f)
print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")



Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```


In [2]:
import json
import csv

DB_FILE = "students_db.json"
STUDENT_FIELDS = ["name", "age", "email", "grade"]

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            students = json.load(file)
        if isinstance(students, list) and all(isinstance(student, dict) for student in students):
            return students
        print("Student database must contain a list of objects.")
    except FileNotFoundError:
        return []
    except json.JSONDecodeError:
        print("Student database contains invalid JSON; starting with an empty list.")
    return []

def save_db(filename, students):
    """Save student records to JSON."""
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(students, file, indent=2)

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    if not isinstance(name, str) or not name.strip():
        return False, "Name must not be empty."
    try:
        age_value = int(age)
    except (TypeError, ValueError):
        return False, "Age must be a positive integer."
    if age_value <= 0 or str(age_value) != str(age).strip():
        return False, "Age must be a positive integer."
    if not isinstance(email, str) or "@" not in email.strip():
        return False, "Email must contain '@'."
    if not isinstance(grade, str) or grade.strip().upper() not in {"A", "B", "C", "D", "F"}:
        return False, "Grade must be A, B, C, D, or F."
    return True, ""

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    name = input("Name: ").strip()
    age = input("Age: ").strip()
    email = input("Email: ").strip()
    grade = input("Grade (A/B/C/D/F): ").strip().upper()
    valid, message = validate_student(name, age, email, grade)
    if not valid:
        print(message)
        return
    students.append({"name": name, "age": int(age), "email": email, "grade": grade})
    print(f"Added {name}.")

def list_students(students):
    """Display all students in a formatted table."""
    if not students:
        print("No students found.")
        return
    print(f"{'#':<4} {'Name':<22} {'Age':<6} {'Email':<30} {'Grade':<6}")
    print("-" * 72)
    for index, student in enumerate(students, start=1):
        print(f"{index:<4} {student.get('name', ''):<22} {student.get('age', '')!s:<6} {student.get('email', ''):<30} {student.get('grade', ''):<6}")

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    term = input("Name to search for: ").strip().casefold()
    matches = [student for student in students if term in student.get("name", "").casefold()]
    if matches:
        list_students(matches)
    else:
        print("No matching students found.")

def update_student(students):
    """Update a student's grade or email by name."""
    term = input("Student name: ").strip().casefold()
    matches = [student for student in students if term in student.get("name", "").casefold()]
    if not matches:
        print("No matching students found.")
        return
    for index, student in enumerate(matches, start=1):
        print(f"{index}. {student.get('name', '')} â€” {student.get('email', '')} â€” {student.get('grade', '')}")
    try:
        selected = int(input("Select student number: ")) - 1
        if selected not in range(len(matches)):
            raise ValueError
    except ValueError:
        print("Invalid student number.")
        return
    student = matches[selected]
    field = input("Update [G]rade or [E]mail? ").strip().lower()
    if field == "g":
        value = input("New grade (A/B/C/D/F): ").strip().upper()
        valid, message = validate_student(student.get("name", ""), student.get("age"), student.get("email", ""), value)
        if not valid:
            print(message)
            return
        student["grade"] = value
    elif field == "e":
        value = input("New email: ").strip()
        valid, message = validate_student(student.get("name", ""), student.get("age"), value, student.get("grade", ""))
        if not valid:
            print(message)
            return
        student["email"] = value
    else:
        print("Choose G or E.")
        return
    print("Student updated.")

def delete_student(students):
    """Remove a student by name."""
    term = input("Student name to delete: ").strip().casefold()
    matches = [student for student in students if term in student.get("name", "").casefold()]
    if not matches:
        print("No matching students found.")
        return
    for index, student in enumerate(matches, start=1):
        print(f"{index}. {student.get('name', '')}")
    try:
        selected = int(input("Select student number: ")) - 1
        if selected not in range(len(matches)):
            raise ValueError
    except ValueError:
        print("Invalid student number.")
        return
    student = matches[selected]
    if input(f"Delete {student.get('name', '')}? [y/N] ").strip().lower() == "y":
        students.remove(student)
        print("Student deleted.")
    else:
        print("Deletion cancelled.")

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    added = 0
    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            if not reader.fieldnames or not set(STUDENT_FIELDS).issubset(reader.fieldnames):
                print(f"Skipped import: CSV must include columns {STUDENT_FIELDS}.")
                return
            for row_number, row in enumerate(reader, start=2):
                name, age, email, grade = (row.get(field) for field in STUDENT_FIELDS)
                valid, message = validate_student(name, age, email, grade)
                if not valid:
                    print(f"Skipped row {row_number}: {message}")
                    continue
                students.append({"name": name.strip(), "age": int(age), "email": email.strip(), "grade": grade.strip().upper()})
                added += 1
    except FileNotFoundError:
        print(f"File not found: {csv_filename}")
        return
    except csv.Error as error:
        print(f"Could not read CSV: {error}")
        return
    print(f"Imported {added} student(s).")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    try:
        with open(csv_filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=STUDENT_FIELDS)
            writer.writeheader()
            writer.writerows({field: student.get(field, "") for field in STUDENT_FIELDS} for student in students)
        print(f"Exported {len(students)} student(s) to {csv_filename}.")
    except (OSError, csv.Error) as error:
        print(f"Could not export CSV: {error}")

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

# run_student_db()  # Uncomment to test


In [ ]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")